# Checkpoint I · Examen blanc — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/checkpoints/partie_1/03_examen_solutions.ipynb)

Les solutions des deux questions de code (CP1.3 et CP1.5), exécutées, puis l'enregistrement des réponses que vérifie la partie B du notebook de l'examen (cellules marquées `answer`, récoltées par `tools/build_answers.py`). Le corrigé détaillé et le barème sont dans `03_examen_corrige.md`.

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        if pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.13 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Partie A · Les deux questions de code

In [2]:
# Data of the two code questions (run this cell first)
import numpy as np

penguins = wb.datasets.load_penguins(dropna=True)          # 333 penguins without missing values
MEASURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
X = penguins[MEASURES].to_numpy(dtype=float)               # shape (333, 4): one row per penguin
gentoo_mass = penguins.loc[penguins["species"] == "Gentoo", "body_mass_g"].to_numpy(dtype=float)
print("X:", X.shape, "· gentoo_mass:", gentoo_mass.shape)

X: (333, 4) · gentoo_mass: (119,)


### CP1.3 — Partie 0 : la moyenne qui oublie `axis` 🐛 ★ ⏱️ 5 min · 1 point
**Objectif :** diagnostiquer et corriger une standardisation qui mélange les colonnes.

Un collègue veut standardiser les quatre mesures des manchots **colonne par colonne** : dans chaque colonne, retirer la moyenne de la colonne puis diviser par son écart-type (ddof = 0, comme `np.std`). Exécute son code (cellule suivante) : les moyennes des colonnes standardisées devraient valoir 0, et leurs écarts-types 1.

a) Que calcule `X.mean()` ? Quelle est la forme (*shape*) de son résultat ? Pourquoi la standardisation du collègue est-elle fausse ? (réponds sur ta feuille) **[0,3 pt]**
b) Écris `standardize(X)`, qui renvoie la version standardisée de `X` colonne par colonne (ddof = 0), sans modifier `X`. **[0,4 pt]**
c) Avec ta fonction, donne le z-score de la masse du premier manchot (ligne 0, colonne `body_mass_g`), à 2 décimales (sur ta feuille). **[0,3 pt]**

In [3]:
# The colleague's code (do not change this cell: run it and read what it prints)
X_std_bug = (X - X.mean()) / X.std()
print("means of the columns:", X_std_bug.mean(axis=0).round(2))
print("stds of the columns :", X_std_bug.std(axis=0).round(3))

means of the columns: [-0.59 -0.6  -0.5   1.69]
stds of the columns : [0.003 0.001 0.008 0.439]


In [4]:
# a) X.mean() averages ALL the values of the table (333 x 4 = 1 332 numbers, millimetres and grams mixed):
#    one number, of shape (). The colleague removes the same number from every column and divides by one global std.
print("X.mean():", X.mean(), "· shape:", np.shape(X.mean()), "· X.std():", X.std())
print("X.mean(axis=0):", X.mean(axis=0).round(1), "· shape:", X.mean(axis=0).shape)


def standardize(X):
    """Return the z-scores of X, column by column: (X - column mean) / column std, with ddof = 0.

    X is a 2-D array of shape (n_samples, n_features); it must not be modified.
    """
    X = np.asarray(X, dtype=float)
    return (X - X.mean(axis=0)) / X.std(axis=0)       # shapes (333, 4) - (4,): broadcasting along the rows


Z_3 = standardize(X)
print("means of the columns:", Z_3.mean(axis=0).round(2) + 0.0)   # + 0.0 turns -0.0 into 0.0
print("stds of the columns :", Z_3.std(axis=0).round(3))
print("z-score of the mass of the first penguin:", round(float(Z_3[0, 3]), 4))

X.mean(): 1117.2954204204204 · shape: () · X.std(): 1829.9730553535119
X.mean(axis=0): [  44.    17.2  201.  4207.1] · shape: (4,)
means of the columns: [0. 0. 0. 0.]
stds of the columns : [1. 1. 1. 1.]
z-score of the mass of the first penguin: -0.5685


In [5]:
wb.record("CP1.3c", float(Z_3[0, 3]), decimals=4,
          mistakes={"c'est avec ddof = 1, la valeur par défaut de pandas `.std()` : l'énoncé demande ddof = 0, comme `np.std`":
                    float((X[0, 3] - X[:, 3].mean()) / X[:, 3].std(ddof=1)),
                    "c'est la standardisation fautive du collègue : la moyenne et l'écart-type de TOUT le tableau, au lieu de ceux de la colonne":
                    float(X_std_bug[0, 3]),
                    "tu as standardisé chaque LIGNE (un manchot, avec ses quatre unités) : la moyenne et l'écart-type se calculent par colonne (axis=0)":
                    float((X[0, 3] - X[0].mean()) / X[0].std()),
                    "c'est une normalisation min-max : la standardisation retire la moyenne de la colonne et divise par son écart-type":
                    float((X[0, 3] - X[:, 3].min()) / (X[:, 3].max() - X[:, 3].min()))})

WB_ANSWER {"decimals": 4, "hash": "5eec47564517d5e2f9d3bc9fbce7933dd1f01e30e58735043996df929a357ea1", "hash_coarse": "6a8eb6f0777d7b81f4c08e421869dfacba02d97517577137dd61e0976aadee4b", "id": "CP1.3c", "kind": "float", "magnitude": -1, "mistakes": {"23e077a061d54d95494e0fa1d818a743f257cdc50004b3ccf0f52a86f6b29691": "c'est avec ddof = 1, la valeur par défaut de pandas `.std()` : l'énoncé demande ddof = 0, comme `np.std`", "40563d34d09b6d8bca46f81f67c32992e0b98f4c62f84a666dcd04107ff50bbe": "tu as standardisé chaque LIGNE (un manchot, avec ses quatre unités) : la moyenne et l'écart-type se calculent par colonne (axis=0)", "8192b0b720236bb3c5f5a78e424782d108d1503e5a6c42c0676ba5e8c1b4164f": "c'est la standardisation fautive du collègue : la moyenne et l'écart-type de TOUT le tableau, au lieu de ceux de la colonne", "9e5acb188101c705c7c7cd2f17b60dae5e2d71a5d597727ab35ba91deb50e1d7": "c'est une normalisation min-max : la standardisation retire la moyenne de la colonne et divise par son écart-t

### CP1.5 — Coder un intervalle de confiance bootstrap 🔨 ★★ ⏱️ 10 min · 1,5 point
**Objectif :** programmer le bootstrap percentile de la moyenne, sans librairie.

a) Écris `bootstrap_ci_mean(x, confidence=0.95, n_boot=1000, seed=0)`, qui renvoie l'intervalle de confiance bootstrap **percentile** de la moyenne de `x`, sous la forme d'un tuple de deux `float` `(bas, haut)`, bornes non arrondies : **[0,8 pt]**
- un seul générateur, créé dans la fonction : `rng = np.random.default_rng(seed)` ;
- pour chacun des `n_boot` rééchantillons, **dans l'ordre**, tire `n` indices avec remise, `idx = rng.integers(0, n, size=n)` ($n$ = taille de `x`), et calcule la moyenne de `x[idx]` (tirer toutes les lignes d'un coup, `size=(n_boot, n)`, ou utiliser `rng.choice(x, size=n)` donne exactement les mêmes nombres) ;
- les bornes sont les percentiles $50\,(1 - c)$ et $50\,(1 + c)$ de ces moyennes, avec $c$ = `confidence` (`np.percentile`).

Puis, sur la masse des manchots Gentoo (`gentoo_mass`, en grammes), et sur ta feuille :
b) l'intervalle à 95 % (graine 0), à 1 décimale ; **[0,2 pt]**
c) l'intervalle à 90 % (graine 0) : est-il plus large ou plus étroit ? Pourquoi ? **[0,2 pt]**
d) Laquelle de ces deux phrases est juste, et qu'est-ce qui ne va pas dans l'autre ? (1) « 95 % des manchots Gentoo pèsent entre … et … g. » (2) « Cette méthode donne un intervalle qui contient la vraie masse moyenne des Gentoo environ 95 fois sur 100. » **[0,3 pt]**

In [6]:
def bootstrap_ci_mean(x, confidence=0.95, n_boot=1000, seed=0):
    """Percentile bootstrap confidence interval of the mean of x, as a tuple (low, high) of floats."""
    x = np.asarray(x, dtype=float)
    n = len(x)
    rng = np.random.default_rng(seed)
    means = np.empty(n_boot)
    for b in range(n_boot):                            # in order: resample b uses the b-th draws of rng
        idx = rng.integers(0, n, size=n)               # n indices drawn WITH replacement
        means[b] = x[idx].mean()
    low, high = np.percentile(means, [50 * (1 - confidence), 50 * (1 + confidence)])
    return float(low), float(high)


ci_95 = bootstrap_ci_mean(gentoo_mass)
ci_90 = bootstrap_ci_mean(gentoo_mass, confidence=0.90)
print(f"sample mean: {gentoo_mass.mean():.1f} g · 95 %: {ci_95[0]:.1f} to {ci_95[1]:.1f} g · "
      f"90 %: {ci_90[0]:.1f} to {ci_90[1]:.1f} g")

sample mean: 5092.4 g · 95 %: 5008.0 to 5185.1 g · 90 %: 5023.7 to 5166.2 g


In [7]:
# Classic wrong procedures of a), each recorded with its message
n_g = len(gentoo_mass)
rng = np.random.default_rng(0)
means_ok = np.array([gentoo_mass[rng.integers(0, n_g, size=n_g)].mean() for _ in range(1000)])     # the right one
assert np.allclose(np.percentile(means_ok, [2.5, 97.5]), ci_95)
rng = np.random.default_rng(0)
means_nboot = np.array([gentoo_mass[rng.integers(0, n_g, size=1000)].mean() for _ in range(1000)])  # size n_boot
rng = np.random.default_rng(0)
means_cols = gentoo_mass[rng.integers(0, n_g, size=(n_g, 1000))].mean(axis=0)                       # shape (n, n_boot)
np.random.seed(0)
means_legacy = np.array([gentoo_mass[np.random.randint(0, n_g, size=n_g)].mean() for _ in range(1000)])


def wrong_procedures(c, right):
    """The interval at confidence c given by each classic wrong procedure (message -> [low, high])."""
    q = [50 * (1 - c), 50 * (1 + c)]
    return {
        "chaque rééchantillon a la taille de x (n tirages avec remise), pas n_boot": np.percentile(means_nboot, q).tolist(),
        "ce sont les percentiles des masses elles-mêmes : l'intervalle porte sur la MOYENNE, prends les percentiles des n_boot moyennes bootstrap":
            np.percentile(gentoo_mass, q).tolist(),
        "tes tirages ont la forme (n, n_boot) : chaque rééchantillon doit utiliser, dans l'ordre, n tirages consécutifs (size=n dans la boucle, ou size=(n_boot, n))":
            np.percentile(means_cols, q).tolist(),
        "np.random.seed et np.random.randint utilisent l'ancien générateur : l'énoncé impose rng = np.random.default_rng(seed) et rng.integers":
            np.percentile(means_legacy, q).tolist(),
        "ta fonction arrondit les bornes : renvoie-les sans arrondir (on n'arrondit qu'en écrivant la réponse)":
            [round(v, 1) for v in right],
    }


wb.record("CP1.5b", list(ci_95), decimals=4,
          mistakes={"pour 95 %, coupe 2,5 % de chaque côté : les percentiles 2,5 et 97,5 de la distribution bootstrap":
                    list(ci_90), **wrong_procedures(0.95, ci_95)})
wb.record("CP1.5c", list(ci_90), decimals=4,
          mistakes={"pour 90 %, coupe 5 % de chaque côté : les percentiles 5 et 95 (utilise bien l'argument confidence)":
                    list(ci_95),
                    "pour 90 %, les percentiles sont 50 (1 − c) = 5 et 50 (1 + c) = 95, pas 100 (1 − c) = 10 et 100 c = 90":
                    np.percentile(means_ok, [10, 90]).tolist(),
                    **wrong_procedures(0.90, ci_90)})

WB_ANSWER {"decimals": 4, "element_coarse": [["b60bd7a5c1a9ac8eebf32b9c4889c260f12e7487a2770415022406276cd54697"], ["e25a2ad85753214baf01fc82fb99d94218d9685588b815416e381e38d1cd0191"]], "element_hashes": ["dd1f77e97e8207323d12343bcebae38215eb11ff3de8a4f04598fc3657fd4b4d", "9e67af340127d785d5b034500c3d1129c8a311b1df66703fbaf4ac6bb8119d76"], "hash": "cc5a8fe0de33351ffcd0129831c3b01457a3639d21fe0a249554691349d10a4c", "id": "CP1.5b", "kind": "array", "mistakes": {"006a7d26d06cb2d955f3b4ef149ea0761b7734975c1c26c41f9994608757f7b2": "pour 95 %, coupe 2,5 % de chaque côté : les percentiles 2,5 et 97,5 de la distribution bootstrap", "01a9bcfbfece03e545c91280bc55614f2360bc25ad485f38e50f45960f70f151": "chaque rééchantillon a la taille de x (n tirages avec remise), pas n_boot", "246912cc5437e5f313a6fe5c463786481151ae25ae824f87f6bb7665cb06061f": "np.random.seed et np.random.randint utilisent l'ancien générateur : l'énoncé impose rng = np.random.default_rng(seed) et rng.integers", "66bcc34fbc15b67bd

## Partie B · Les réponses vérifiées après l'examen

Chaque réponse est calculée à partir des nombres du sujet ; les erreurs classiques reçoivent un message qui dit où chercher.

In [8]:
# The paper questions only use the numbers of their statements (01_examen_sujet.md)
import math
from fractions import Fraction

import numpy as np

u_2, v_2, w_2 = np.array([1, -2, 2]), np.array([2, 3, 4]), np.array([2, 1, 0])          # CP1.2
x_4 = np.array([2, 4, 6, 8, 10], dtype=float)                                           # CP1.4
y_4 = np.array([3, 7, 5, 11, 9], dtype=float)
N_6, PREVALENCE_6, SENSITIVITY_6, SPECIFICITY_6 = 25_000, 0.04, 0.90, 0.92              # CP1.6
SICK_6 = N_6 * PREVALENCE_6
TP_6, FP_6 = SENSITIVITY_6 * SICK_6, (1 - SPECIFICITY_6) * (N_6 - SICK_6)
FN_6, TN_6 = SICK_6 - TP_6, (N_6 - SICK_6) - FP_6
THETA_8, PRIOR_8 = [Fraction(1, 5), Fraction(1, 2), Fraction(4, 5)], [Fraction(1, 4), Fraction(1, 2), Fraction(1, 4)]


def posterior(prior, likelihood):
    """Bayes' rule on a list of hypotheses (exact with fractions)."""
    joint = [p * l for p, l in zip(prior, likelihood)]
    return [j / sum(joint) for j in joint]


POST1_8 = posterior(PRIOR_8, THETA_8)                    # after one heads
POST2_8 = posterior(POST1_8, THETA_8)                    # after heads, heads


def f_9(point):
    x, y = point
    return (x - 1) ** 2 - 0.5 * (y + 2) ** 2


def grad_9(point):
    x, y = point
    return np.array([2 * (x - 1), -(y + 2)])


STEPS_9 = [np.array([2.0, -1.0])]
for _ in range(2):
    STEPS_9.append(STEPS_9[-1] - 0.25 * grad_9(STEPS_9[-1]))
P_11 = np.array([0.4, 0.3, 0.2, 0.1])                    # vowels, frequent, medium, rare consonants
Q_11 = np.full(4, 0.25)
H_11 = float(-(P_11 * np.log2(P_11)).sum())
LENGTHS_11 = np.array([1, 2, 3, 3])                       # Huffman lengths [V, F, M, R]

**CP1.1 — Questions flash (le verdict seulement ; les justifications se notent avec le corrigé)**

In [9]:
wb.record("CP1.1a", True)
wb.record("CP1.1b", False)
wb.record("CP1.1c", True)
wb.record("CP1.1d", True)
wb.record("CP1.1e", False)
wb.record("CP1.1f", False)

WB_ANSWER {"hash": "8f35ee50d1ca708638280f95cab0a03a353052a0b9c57023055770427e3d5540", "id": "CP1.1a", "kind": "bool"}
📝 Ex CP1.1a : réponse enregistrée (bool).
WB_ANSWER {"hash": "c398e04cf48806d09bf9ffd21f933f269638b224787e46b2884534ac2c9c2f54", "id": "CP1.1b", "kind": "bool"}
📝 Ex CP1.1b : réponse enregistrée (bool).
WB_ANSWER {"hash": "a7f16df0bff1d52125f1d06e1d4f4636a521a6f7113ba5e98e8f06501db84268", "id": "CP1.1c", "kind": "bool"}
📝 Ex CP1.1c : réponse enregistrée (bool).
WB_ANSWER {"hash": "2d500e461c56903449a8eb131da9da6263055c01b4d0f853748bb0a7c0335e57", "id": "CP1.1d", "kind": "bool"}
📝 Ex CP1.1d : réponse enregistrée (bool).
WB_ANSWER {"hash": "43f7160536b6d473ac1eb859d66b34a996832217c2b0d4e78898ef037a57a235", "id": "CP1.1e", "kind": "bool"}
📝 Ex CP1.1e : réponse enregistrée (bool).
WB_ANSWER {"hash": "47586fb83233990e4cb13fb0cdeb33e31f774fa2c5d4176c62c6e369db18bbd1", "id": "CP1.1f", "kind": "bool"}
📝 Ex CP1.1f : réponse enregistrée (bool).


**CP1.2 — Norme, produit scalaire et log₂**

In [10]:
wb.record("CP1.2a", int(round(float(np.linalg.norm(u_2)))), mistakes={"c'est ‖u‖² : il manque la racine carrée": 9,
                     "la norme n'est pas la somme des valeurs absolues : c'est la racine de la somme des carrés": 5})
wb.record("CP1.2b", int(u_2 @ v_2), mistakes={"attention au signe : la deuxième composante de u est négative": 16})
wb.record("CP1.2c", bool(u_2 @ w_2 == 0))
wb.record("CP1.2d", int(round(math.log2(1 / 32))))
wb.record("CP1.2e", math.log2(24), decimals=3, mistakes={"24 = 8 × 3 : le log d'un produit est la SOMME des logs, pas un produit": 3 * math.log2(3)})

WB_ANSWER {"hash": "f38a63eb97f75074239db6c921b0ab4275b004bbff15f5d6e280ec8519d90202", "id": "CP1.2a", "kind": "int", "magnitude": 0, "mistakes": {"07ef25c69a5e9443bdc7135effe69547a420b80ce86a02d3df97896d9f656b8b": "la norme n'est pas la somme des valeurs absolues : c'est la racine de la somme des carrés", "0851cf0364cf98fcd4a79de4b715a6fc58a58d8d434d4cb4e7b4054c335cf435": "c'est ‖u‖² : il manque la racine carrée"}, "sign": 1}
📝 Ex CP1.2a : réponse enregistrée (int).
WB_ANSWER {"hash": "acd716ac43c608c889fe5bcefa37d376d71b5b96bc1a46d995f947741a4e4674", "id": "CP1.2b", "kind": "int", "magnitude": 0, "mistakes": {"53c9e44ce1e1ca4324f2e8bd64cbc57c3d583aab8baf7e3a6ed41cd39fbd5ff8": "attention au signe : la deuxième composante de u est négative"}, "sign": 1}
📝 Ex CP1.2b : réponse enregistrée (int).
WB_ANSWER {"hash": "e5fb6ecd30f387e18f92e63e567d2c0528cb5cec54d9c5d39f1087a8f341040b", "id": "CP1.2c", "kind": "bool"}
📝 Ex CP1.2c : réponse enregistrée (bool).
WB_ANSWER {"hash": "a2494d49a6374a

**CP1.4 — Statistiques de cinq points à la main**

In [11]:
wb.record("CP1.4a", [x_4.mean(), y_4.mean()], decimals=1, mistakes={"tu as inversé x̄ et ȳ : l'ordre demandé est [x̄, ȳ]": [y_4.mean(), x_4.mean()]})
wb.record("CP1.4b", [x_4.var(), x_4.std()], decimals=2, mistakes={"tu as divisé par n − 1 : ici ddof = 0, on divise par n": [x_4.var(ddof=1), x_4.std(ddof=1)],
                     "l'ordre demandé est [variance, écart-type]": [x_4.std(), x_4.var()],
                     "tu n'as pas divisé la somme des carrés des écarts par n": [((x_4 - x_4.mean()) ** 2).sum(), math.sqrt(((x_4 - x_4.mean()) ** 2).sum())]})
wb.record("CP1.4c", (10 - x_4.mean()) / x_4.std(), decimals=2, mistakes={"tu as pris l'écart-type avec ddof = 1 : ici ddof = 0, on divise par n": (10 - x_4.mean()) / x_4.std(ddof=1),
                     "on divise l'écart à la moyenne par l'écart-type, pas par la variance": (10 - x_4.mean()) / x_4.var()})
wb.record("CP1.4d", [np.cov(x_4, y_4, ddof=0)[0, 1], np.cov(x_4, y_4, ddof=1)[0, 1]], decimals=1, mistakes={"tu as inversé les deux : avec ddof = 0, on divise par n = 5 ; avec ddof = 1, par n − 1 = 4": [np.cov(x_4, y_4, ddof=1)[0, 1], np.cov(x_4, y_4, ddof=0)[0, 1]]})
wb.record("CP1.4e", np.corrcoef(x_4, y_4)[0, 1], decimals=2, mistakes={"c'est r² : la corrélation est la covariance divisée par le produit des écarts-types": np.corrcoef(x_4, y_4)[0, 1] ** 2,
                     "divise la covariance par le PRODUIT des écarts-types, pas par la somme des carrés des écarts": np.cov(x_4, y_4, ddof=0)[0, 1] / ((x_4 - x_4.mean()) ** 2).sum()})
wb.record("CP1.4f", [np.append(y_4, 40).mean(), np.median(np.append(y_4, 40))], decimals=1, mistakes={"avec six valeurs, la médiane est la moyenne des deux valeurs du milieu, une fois triées": [np.append(y_4, 40).mean(), 7.0]})

WB_ANSWER {"decimals": 1, "element_hashes": ["51e0a41b748ef055166f38bb073d18ce85fe44562f0cf13b1c5251aa9b72f4f9", "c0b69c28b43832aa770b249c0e26dd1c68cea4df12265693121d1f837cbba506"], "hash": "bf2fe2fbeb43bf2e7d3d46425f145c67322b5ab2a8cf7a91fd748b3cf4376513", "id": "CP1.4a", "kind": "array", "mistakes": {"e67df556a500235ab07ae24bb945ad734ef37ac5da2fcd6ddbb127125fb12ec0": "tu as inversé x̄ et ȳ : l'ordre demandé est [x̄, ȳ]"}, "shape": [2]}
📝 Ex CP1.4a : réponse enregistrée (array, 1 décimale(s)).
WB_ANSWER {"decimals": 2, "element_coarse": [["a67b444b132b4b89cbe3da599cc6d8436d71b3474c0e92743c8df6ae66249a4f"], ["d6ad673b56b80b9510baac79679a1e3208c844d3cfef7d9a77d99bf64b4a0a8b"]], "element_hashes": ["fa6af5b10afc376997552a1041de85926a7eda746ec963413c432f790eaa6bd8", "7e324f91a3417545e66410b80679be105c0c07ba3a9908e79b273f164897ee6f"], "hash": "e23f1213186b8af51caf4e5d3764d0587072fe82acc21b88330ce70f78450f19", "id": "CP1.4b", "kind": "array", "mistakes": {"117e0a4615822262722bac9cf3a313a2882

WB_ANSWER {"decimals": 1, "element_hashes": ["c02862f3c640399e803c666a82c0243287d0afb3b91722d3aff92e4dafb8f8fe", "d4f206d11eeec9da3fa772fbf22976dde74bdb0c0ca0cb1d314b2a58a5e8d9ab"], "hash": "f0300759d0f806676baedcb63a29a9d09a69ff3cb1d8d573458e9ecfb57d73b5", "id": "CP1.4d", "kind": "array", "mistakes": {"689fda74d0f3b0cb6a64bef4c53cad48568efe5a6c38239b33aea6d7bca3325c": "tu as inversé les deux : avec ddof = 0, on divise par n = 5 ; avec ddof = 1, par n − 1 = 4"}, "shape": [2]}
📝 Ex CP1.4d : réponse enregistrée (array, 1 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "093e0f7406336bcfb673f66e8d43e748f1b01ef362b19896d8036e77c2416c9f", "hash_coarse": "8d9bb68f8bb26a0f5f4e12d0460f78e407c675e50c19882e58eccbb780bdae81", "id": "CP1.4e", "kind": "float", "magnitude": -1, "mistakes": {"a8ac581032309006fda8abb04792dd60a0a1e58222047d76c45fe499e9f2f16d": "divise la covariance par le PRODUIT des écarts-types, pas par la somme des carrés des écarts", "f8ac9d50e6d20a7c1411008dc8c3545d0027ea486ff6912d

**CP1.6 — Dépistage : matrice de confusion, precision, NPV et règle de Bayes**

In [12]:
wb.record("CP1.6a", [int(round(TP_6)), int(round(FN_6)), int(round(FP_6)), int(round(TN_6))], mistakes={"les faux positifs se comptent parmi les SAINS seulement, pas parmi toutes les personnes": [900, 100, 2000, 22000],
                     "l'ordre demandé est TP, FN, FP, TN": [900, 1920, 100, 22080]})
wb.record("CP1.6b", TP_6 / (TP_6 + FP_6), decimals=3, mistakes={"c'est la sensibilité P(positif | malade) : on demande P(malade | positif)": 0.9,
                     "c'est P(malade et positif) : divise par P(positif)": SENSITIVITY_6 * PREVALENCE_6})
wb.record("CP1.6c", TN_6 / (TN_6 + FN_6), decimals=4, mistakes={"c'est la spécificité P(négatif | sain) : on demande P(sain | négatif)": 0.92})
wb.record("CP1.6d", (TP_6 + TN_6) / N_6, decimals=3, mistakes={"c'est la moyenne de la sensibilité et de la spécificité (la balanced accuracy) : l'accuracy compte les bonnes réponses parmi les 25 000": 0.91})
wb.record("CP1.6f", SENSITIVITY_6 * (TP_6 / (TP_6 + FP_6)) / (SENSITIVITY_6 * (TP_6 / (TP_6 + FP_6)) + (1 - SPECIFICITY_6) * (FP_6 / (TP_6 + FP_6))), decimals=3, mistakes={"un second résultat positif doit faire monter la probabilité : applique la règle de Bayes une seconde fois": TP_6 / (TP_6 + FP_6),
                     "c'est P(deux positifs | malade) : on demande P(malade | deux positifs)": SENSITIVITY_6 ** 2})

WB_ANSWER {"decimals": 0, "element_hashes": ["1c75aa475dfa1fdf9f68689e47ec7e61b138739d210e74e9e072f3909b6ff9cf", "72b5241ee3d726924ede64a3b1e07042a9c15bb9d1e33562f811e3d88ac52bdb", "48884afe954a8a6a47bde6b3f1a81bb0e372656475debf8222a7175e58d8566f", "fa39e9ab9e294f31d6efea51a09b6433a10e61ed9474eb25a9a2e7302a72990e"], "hash": "e998b319c0c93efd1e09548c5e7cdda00eccb87e51601a533fea96c2978597db", "id": "CP1.6a", "integer": true, "kind": "array", "mistakes": {"7a2590e6276e77e9e38372e82f98ba43362d27f8a3454042fd2923c5856284c6": "les faux positifs se comptent parmi les SAINS seulement, pas parmi toutes les personnes", "987ae7e62d00999d03002a1db77185dde4aece04a83d3b5fe0b298c896a3c56a": "l'ordre demandé est TP, FN, FP, TN"}, "shape": [4]}
📝 Ex CP1.6a : réponse enregistrée (array, 0 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "db2d0a66816c258d4a31678af39c515b04665f0b02e5836e7a2238d2dd564709", "hash_coarse": "483f6edf294086dca22e2e2e07dacd5ec2cd8877bbc2b03a1e0fcb133dd1db91", "id": "CP1.6b", "kin

**CP1.7 — Lire une courbe ROC et une courbe PR (seule la ligne de base se vérifie ici : les lectures se notent avec le corrigé)**

In [13]:
wb.record("CP1.7e", 0.01, decimals=2, mistakes={"la diagonale vaut pour la courbe ROC ; un classifieur au hasard a une precision égale à la proportion de positifs": 0.5,
                     "0 serait la precision d'un modèle dont aucune alerte n'est juste : au hasard, la part des fraudes parmi les alertes est la même que partout": 0.0})

WB_ANSWER {"decimals": 2, "hash": "9be568296083f1b6f2b6940700b5c56bf9bf79518287ed5a5a6a1be74e016912", "hash_coarse": "ca6d3716b6ff53cb883dc4ab898af4888493d6d77ff8710cb937905dc53cbec8", "id": "CP1.7e", "kind": "float", "magnitude": -2, "mistakes": {"9df4082482ccb1d44fb95c9838deab3b897d23b5d2761bcaee278805d9bdeca2": "la diagonale vaut pour la courbe ROC ; un classifieur au hasard a une precision égale à la proportion de positifs", "c375b6c2677649c6dfd94a68b8c692817f5800e6bff1042a6a6b36bf15444848": "0 serait la precision d'un modèle dont aucune alerte n'est juste : au hasard, la part des fraudes parmi les alertes est la même que partout"}, "sign": 1}
📝 Ex CP1.7e : réponse enregistrée (float, 2 décimale(s)).


**CP1.8 — Trois hypothèses, deux lancers**

In [14]:
wb.record("CP1.8a", [float(p) for p in POST1_8], decimals=2, mistakes={"tu n'as pas divisé par l'évidence : un posterior somme à 1": [float(p * t) for p, t in zip(PRIOR_8, THETA_8)],
                     "tu as oublié le prior : multiplie chaque vraisemblance par son prior avant de normaliser": [float(t / sum(THETA_8)) for t in THETA_8]})
wb.record("CP1.8b", [float(p) for p in POST2_8], decimals=3, mistakes={"tu n'as pas divisé par l'évidence : un posterior somme à 1": [float(p * t) for p, t in zip(POST1_8, THETA_8)],
                     "tu as oublié le prior : multiplie chaque vraisemblance par son prior avant de normaliser": [float(t * t / sum(s * s for s in THETA_8)) for t in THETA_8],
                     "prior × θ² est juste, mais il faut encore diviser par leur somme : un posterior somme à 1": [float(p * t * t) for p, t in zip(PRIOR_8, THETA_8)]})
wb.record("CP1.8c", float(sum(p * t * t for p, t in zip(PRIOR_8, THETA_8))), decimals=3, mistakes={"c'est l'évidence du second lancer seulement : celle de la suite est le produit des deux évidences (ou Σ prior × θ²)": float(sum(p * t for p, t in zip(POST1_8, THETA_8))),
                     "c'est l'évidence du premier lancer seulement : il faut celle de la suite des deux": float(sum(p * t for p, t in zip(PRIOR_8, THETA_8)))})
wb.record("CP1.8d", 0.8, decimals=2, choices=[0.2, 0.5, 0.8], mistakes={"0,5 avait le plus grand PRIOR : le MAP se lit sur le posterior après les deux lancers": 0.5,
                     "relis le posterior de b) : le MAP est l'hypothèse dont le posterior est le plus grand": 0.2})
wb.record("CP1.8e", float(sum(p * t for p, t in zip(POST2_8, THETA_8))), decimals=3, mistakes={"ta valeur est l'un des θ : la probabilité de face au troisième lancer pondère chaque θ par son posterior": 0.8,
                     "c'est la prédiction faite après UNE face : pour le troisième lancer, le prior est le posterior après les deux faces": float(sum(p * t for p, t in zip(POST1_8, THETA_8)))})

WB_ANSWER {"decimals": 2, "element_coarse": [["5889b240219d97e9d04d91af1a0bfc9fa5955ef56953a5dab0799089b9fe04a5"], ["cde4c74b0a0c0ed106076404b5f889a5b9962c882eb39f08d43fb0ef64fbd1cb"], ["c94bcdc3099b5744f87296b01a7d3accdaeb69e86a16deaed19f169770299732"]], "element_hashes": ["11464d1f8f9e84d8cf4d1df661f350fadae4a921131a77d16b3d59bdc8b68260", "2e7934cddc7c7a9081fc4971f8460b927a8567cbb1517443fa9564032fd66bba", "c5f444ac9ba9b7fe3b3a0fa3a0502c65ee5906191ae149328078c7a91b54cc07"], "hash": "ed1424043cfe35a7ebcd5db39fe6f1623b8d8b792abfc6fa2363cf57efb423ad", "id": "CP1.8a", "kind": "array", "mistakes": {"519a07a65c9f7b2200366a94c2c25a70ab7edced8377d55d47879c857707d77c": "tu n'as pas divisé par l'évidence : un posterior somme à 1", "b29a18a90b90934e783e118af8c0a6c0a308485d45ffb00fa63b1d87d6f90e7d": "tu as oublié le prior : multiplie chaque vraisemblance par son prior avant de normaliser"}, "shape": [3]}
📝 Ex CP1.8a : réponse enregistrée (array, 2 décimale(s)).
WB_ANSWER {"decimals": 3, "element_

**CP1.9 — Gradient, point selle et deux pas de descente**

In [15]:
wb.record("CP1.9b", [1, -2], mistakes={"attention au signe : vérifie ta valeur de y dans l'équation ∂f/∂y = 0": [1, 2]})
wb.record("CP1.9d", [STEPS_9[1].tolist(), STEPS_9[2].tolist()], decimals=4, mistakes={"tu es monté : la descente RETIRE η × gradient": [[2.5, -1.25], [3.25, -1.4375]],
                     "tes pas déplacent y dans le mauvais sens : vérifie le signe de ∂f/∂y": [[1.5, -1.25], [1.25, -1.4375]]})
wb.record("CP1.9e", [f_9(p) for p in STEPS_9], decimals=3)

WB_ANSWER {"decimals": 0, "element_hashes": ["df36c560adde5d4422b2799f24fe79872c386d3a72a87058ef0328affe6e16db", "a2df25161034554118ed3396f3aec1385fa389f65834deff48242052365c0cd1"], "hash": "a080c035483cb15a94f810564bfcab4cf6b385c60440b9778991c5e8365ce8af", "id": "CP1.9b", "integer": true, "kind": "array", "mistakes": {"4a1234b9bf125b2606238fe536df1be2eb2ab5c8b8dfaefb64d212ae2c6485c2": "attention au signe : vérifie ta valeur de y dans l'équation ∂f/∂y = 0"}, "shape": [2]}
📝 Ex CP1.9b : réponse enregistrée (array, 0 décimale(s)).
WB_ANSWER {"decimals": 4, "element_coarse": [["18b28887eef2ae8e377e1ca6d18304406a94236356036b53e6b5b2405003a1ee"], ["263459cb36ace65b64a5e82e099d38cb97a20a5fc502fdba56862e9e30b7546e"], ["738734f1958988d353c62865be585c248d350fb3291bcab28e6ce8aa6a37fe70"], ["a8d0f03acfc6413395da331fe5f5f813a9d3d9fb0ac0ba4dc410f24dcfe584bf"]], "element_hashes": ["a5e0c9945d84bb78076fd8bb469a527ce5ead9fbee04a1d1b599581026e63d08", "18d6f647d264bc84d713515b542e6e0627c49d3ec738f787132

**CP1.10 — Prédire l'effet du learning rate**

In [16]:
wb.record("CP1.10a", "CADB", mistakes={"revois le cas η = 0,3 : calcule x₁ et x₂ à partir de x₀ = 1": "DADB",
                     "revois le cas η = 0,4 : calcule x₁ et x₂ à partir de x₀ = 1": "CACB"})
wb.record("CP1.10b", 1 / 3, decimals=3, mistakes={"1/6 amène x à 0 en un seul pas ; la convergence demande seulement |1 − 6η| < 1": 1 / 6,
                     "c'est la borne pour f′(x) = 2x (le facteur 3 oublié) ; ici f′(x) = 6x": 1.0,
                     "f′(x) = 6x : dérive 3x² avant d'écrire x_{t+1}": 2 / 3})

WB_ANSWER {"hash": "66a185c438dea6382bb873c3b6d83e7c2eb4fc634b06ce3d8a1d57506f11dd94", "id": "CP1.10a", "kind": "str", "mistakes": {"13b540c3c54c634131d46797ac93300ca3e3a3c5a34abb77577466596d8385c2": "revois le cas η = 0,3 : calcule x₁ et x₂ à partir de x₀ = 1", "813b6d624ac0ba925d017f6dfec5199eb7e4956890ad8feb4edabcc3f54c1fe3": "revois le cas η = 0,4 : calcule x₁ et x₂ à partir de x₀ = 1"}}
📝 Ex CP1.10a : réponse enregistrée (str).
WB_ANSWER {"decimals": 3, "hash": "6c7d14ec128d82e37476677490a90c72e60636a3a51a2a6c91df40322e74229e", "hash_coarse": "068ae5c9556582a07d55a5b1c85835e7485605344f4df3c44cf689e90a61bb49", "id": "CP1.10b", "kind": "float", "magnitude": -1, "mistakes": {"0e98cf38b240ca9c9049ed8cd6a09bc9bcb83d72fc101b10a28d805b5c64ca64": "f′(x) = 6x : dérive 3x² avant d'écrire x_{t+1}", "b21cd597d439c1be298890274ab846ee573e95d68dbacea74426b065a8cf362e": "c'est la borne pour f′(x) = 2x (le facteur 3 oublié) ; ici f′(x) = 6x", "fcd2963b54cd9d58ba139528320d562733eed1ec45eb419123d788

**CP1.11 — Entropie, cross-entropy, KL et Huffman**

In [17]:
wb.record("CP1.11a", H_11, decimals=3, mistakes={"c'est en nats : on demande des bits (log₂)": float(-(P_11 * np.log(P_11)).sum()),
                     "pondère chaque surprise −log₂ p_i par sa probabilité p_i": float(-np.log2(P_11).sum()),
                     "p n'est pas uniforme : calcule −Σ p_i log₂ p_i avec ses quatre probabilités": 2.0})
wb.record("CP1.11b", LENGTHS_11.tolist(), mistakes={"l'ordre demandé est [V, F, M, R], du plus probable au moins probable": [3, 3, 2, 1],
                     "c'est un code de longueur fixe : Huffman fusionne à chaque étape les deux groupes les moins probables": [2, 2, 2, 2]})
wb.record("CP1.11c", float((P_11 * LENGTHS_11).sum()), decimals=2, mistakes={"pondère chaque longueur par la probabilité de son symbole": float(LENGTHS_11.mean())})
wb.record("CP1.11e", float(-(P_11 * np.log2(Q_11)).sum()), decimals=3, mistakes={"dans H(p, q), les poids viennent de p et les logs de q": H_11,
                     "attention à l'ordre : H(p, q) pondère par p, le premier argument": float(-(Q_11 * np.log2(P_11)).sum())})
wb.record("CP1.11f", float((P_11 * np.log2(P_11 / Q_11)).sum()), decimals=3, mistakes={"attention au sens : dans KL(p ‖ q), les poids viennent de p": float((Q_11 * np.log2(Q_11 / P_11)).sum()),
                     "c'est en nats : on demande des bits (log₂)": float((P_11 * np.log(P_11 / Q_11)).sum())})
wb.record("CP1.11g", float((Q_11 * np.log2(Q_11 / P_11)).sum()), decimals=3, mistakes={"attention au sens : dans KL(q ‖ p), les poids viennent de q": float((P_11 * np.log2(P_11 / Q_11)).sum()),
                     "c'est en nats : on demande des bits (log₂)": float((Q_11 * np.log(Q_11 / P_11)).sum())})
wb.record("CP1.11h", False)

WB_ANSWER {"decimals": 3, "hash": "0c98f6dc61c857a344111bd1b69fa03f9f38f5910f369829c068b3ef61a74ff8", "hash_coarse": "b2246a4d7e9f5e2261bd05b4fd2d50fe930157660fe1324cfc70c8d19b78bd26", "id": "CP1.11a", "kind": "float", "magnitude": 0, "mistakes": {"5da32d38c87ad8fb15d4cb5d2787e4f3809dd567daa53b48e28858253e8da85c": "pondère chaque surprise −log₂ p_i par sa probabilité p_i", "6a42dc99778ba97e0a14d98ee2b39c40e0ffdf2b02d6747a46fa03e18f51dd22": "p n'est pas uniforme : calcule −Σ p_i log₂ p_i avec ses quatre probabilités", "d7599c0ef0c23e8dc294815b5311f59e9437ad31bbf8d6bb7fbde90050757df5": "c'est en nats : on demande des bits (log₂)"}, "sign": 1}
📝 Ex CP1.11a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 0, "element_hashes": ["78fe9c21ffb9cf0f8d044dfe8faa422e694cf86f1b591f80680315cfe1a7a118", "0eaa6b68ff6f70b937f715740fddd44f61a159e5fe4622ad727afe1e6627f4aa", "2558b39d0fc1a5d897c660086db779b6c2979fd321dee13c351b4a890a24d4fb", "07e111314aa7491187cba7cc5e517e1e680fdaf046